# Emy v2 — Quickstart

Memory-first agentic RAG core with Ollama, web search, reflection scoring, and Markdown vault.

This notebook works on **Colab** (installs Ollama automatically) or locally.

## 1) Setup — Install Ollama + Emy

In [ ]:
import os, subprocess, time, platform

print("Machine:", platform.machine())
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Clean up any bad prior download
    !rm -f /usr/local/bin/ollama ./ollama

    # Detect architecture
    arch = platform.machine().lower()
    if arch in ("x86_64", "amd64"):
        url = "https://ollama.com/download/ollama-linux-amd64.tar.zst"
    elif arch in ("aarch64", "arm64"):
        url = "https://ollama.com/download/ollama-linux-arm64.tar.zst"
    else:
        raise RuntimeError(f"Unsupported architecture: {arch}")

    # Install zstd support if needed, then download and extract
    !apt-get update -qq
    !apt-get install -y -qq zstd

    !wget -O /tmp/ollama.tar.zst "$url"
    !tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C /usr/local

    # Verify binary exists
    !ls -l /usr/local/bin/ollama
    !/usr/local/bin/ollama --version
else:
    print("Local environment detected — assuming Ollama is already installed.")
    print("If not, install from https://ollama.com")

In [ ]:
# Start Ollama server (Colab) or verify it's running (local)
if IN_COLAB:
    proc = subprocess.Popen(
        ["/usr/local/bin/ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)
    print("Ollama started, pid =", proc.pid)

# Install Emy
!pip install git+https://github.com/FallingObject/emy -q

In [ ]:
# Pull the required models (runs once, cached afterward)
!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text

In [ ]:
from pathlib import Path
from emy import Emy, EmyConfig

config = EmyConfig(
    workdir=Path("./runtime"),
    llm_model="qwen2.5:7b",
    embedding_model="nomic-embed-text",
    mode="train",
)
emy = Emy(config)

# Check Ollama is reachable
print("Ollama healthy:", emy.llm.health_check())
print("Available models:", emy.llm.available_models())

## 3) Ingest local documents

In [ ]:
count = emy.ingest("./data/sample_docs")
print(f"Ingested {count} chunks")

## 4) Chat — Agent loop with tool calling

In [ ]:
# Simple greeting (no tools needed)
resp = emy.respond("hello")
print(resp.answer)
print("Trace:", resp.trace)

In [ ]:
# Research question — triggers tool calls
resp = emy.respond("What does the documentation say about memory types?")
print(resp.answer)
print("\nTools used:", resp.tools_used)
if resp.score:
    print(f"Score: {resp.score.overall:.2f} — {resp.score.lesson}")

In [ ]:
# Web search question
resp = emy.respond("What are the latest developments in local LLM agents?")
print(resp.answer)
print("\nTools used:", resp.tools_used)

## 5) Train Emy — Add facts, reflections, and intent examples

In [ ]:
# Store facts the agent should remember
emy.add_fact("preferences", "User prefers concise, technical answers")
emy.add_fact("constraints", "Always cite sources when using retrieved information")

# Store reflections (learned lessons)
emy.add_reflection("Skip retrieval for greeting-only messages to reduce latency")
emy.add_reflection("Use web search when docs lack recent data")

# Add training examples for intent routing
emy.add_training_example("compare these two approaches", "research")
emy.add_training_example("what does slide 5 say", "document_query")
emy.add_training_example("thanks!", "smalltalk")

print("Training data saved to Markdown vault.")

## 6) Inspect the Markdown vault

In [ ]:
# The vault is human-readable Markdown
for name in ["facts.md", "reflections.md", "intents.md"]:
    path = config.vault_dir / name
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    if path.exists():
        print(path.read_text())
    else:
        print("  (not yet created)")

## 7) Train via API — Connect a stronger LLM

Emy exposes a FastAPI training endpoint so you can connect a stronger LLM (Claude, GPT-4, etc.) to teach Emy automatically.

In [ ]:
import httpx

# Example: programmatic training via the API
# (Start the server first: emy serve --mode train)
API = "http://localhost:7860"

# Bulk training
training_data = {
    "examples": [
        {"text": "summarize the paper", "label": "research"},
        {"text": "good morning", "label": "smalltalk"},
    ],
    "facts": [
        {"category": "domain", "content": "This project focuses on agentic RAG systems"},
    ],
    "reflections": [
        {"lesson": "Web fetch is slow — prefer web search snippets when possible"},
    ],
}

# Uncomment to run (requires server running):
# resp = httpx.post(f"{API}/api/train/bulk", json=training_data)
# print(resp.json())

## 8) Deploy mode — Stable, scored responses

In [ ]:
emy.set_mode("deploy")

resp = emy.respond("What are the key components of an agentic RAG system?")
print(resp.answer)
print(f"\nTools: {resp.tools_used}")
if resp.score:
    print(f"Score: {resp.score.overall:.2f}")

## 9) System status

In [ ]:
import json
print(json.dumps(emy.status(), indent=2))

## 10) Launch full UI + API server

```bash
# Terminal:
emy serve --mode train --port 7860

# Then open http://localhost:7860 for Gradio UI
# API docs at http://localhost:7860/docs
```

Or launch the Streamlit flagship UI:
```bash
streamlit run emy/streamlit_app.py
```

Or from notebook:

In [ ]:
# Launch Gradio UI in-notebook (blocking — run in last cell)
from emy.ui import build_ui
demo = build_ui(emy)
demo.launch(share=IN_COLAB if 'IN_COLAB' in dir() else False)